# Part 6 · Notebook 08 — Portfolio Greeks, scenarios and delta hedging

**Sessions:** S8 (Portfolio Greeks, scenario risk & delta hedging) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Turn per-option Greeks into dollar Greeks for a book, with multipliers.
2. Compare the delta–gamma–vega approximation with full revaluation, and see it fail for a crash.
3. Size the stock hedge that flattens a book's delta.
4. Simulate delta hedging: where the P&L comes from, and what hedging costs.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

## 1. A small book

SPY-like underlying at 600, 30-day options with a multiplier of **100** (from contract details, never hard-coded in real code).

In [ ]:
S = 600.0
book = [p.Position("short 20 × 560 puts", "option", -20, K=560.0, T=30 / 365, iv=0.24, cp=-1),
        p.Position("long 10 × 620 calls", "option", 10, K=620.0, T=30 / 365, iv=0.15, cp=1),
        p.Position("short 5 × 600 calls", "option", -5, K=600.0, T=30 / 365, iv=0.18, cp=1),
        p.Position("long 300 shares", "stock", 300, multiplier=1.0)]
pd.DataFrame([vars(b) for b in book])

## 2. Dollar Greeks

Per-contract Greeks are per one share of underlying. For a position multiply by `qty × multiplier`:
* **$delta** = Δ · qty · mult · S: the dollar exposure, the P&L of a 100% move if it stayed linear;
* **vega per point** = (vega/100) · qty · mult: P&L if implied vol rises one point.

`p.to_display(p.greeks(...))` gives vega per point. (The reference also reports $gamma for a 1% move and theta per day.)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def dollar_delta_vega(pos, S, r=0.045, q=0.013):
    if pos.kind == "stock":
        return pos.qty * pos.multiplier * S, 0.0
    g = p.to_display(p.greeks(S, pos.K, pos.T, r, q, pos.iv, pos.cp))
    m = pos.qty * pos.multiplier
    return ..., ...                               # ✍️ ($delta, vega per point)

mine = [p.attempt(dollar_delta_vega, b, S) for b in book]
ref = [(p.dollar_greeks(b, S)["$delta"], p.dollar_greeks(b, S)["vega/pt"]) for b in book]
mine = p.check("dollar delta and vega", mine, ref)
report = pd.DataFrame([p.dollar_greeks(b, S) for b in book], index=[b.name for b in book])
report.loc["book"] = report.sum()
report.round(0)

## 3. Taylor vs full revaluation

The quick risk estimate is a Taylor expansion: `ΔV ≈ Δ·ΔS + ½Γ·ΔS² + vega·Δσ`, summed over positions with quantities and multipliers (raw units: `ΔS = S·dS` in price, `Δσ` as a decimal). A stock position contributes only `qty·mult·ΔS`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def taylor_pnl(book, S, dS, dvol, r=0.045, q=0.013):
    total = 0.0
    for pos in book:
        m = pos.qty * pos.multiplier
        x = S * dS
        if pos.kind == "stock":
            total += m * x
            continue
        g = p.greeks(S, pos.K, pos.T, r, q, pos.iv, pos.cp)
        total += ...                              # ✍️ m × (delta·x + ½·gamma·x² + vega·dvol)
    return float(total)

shocks = [(0.01, 0.0), (-0.05, 0.03), (-0.20, 0.25), (0.10, -0.05)]
mine = [p.attempt(taylor_pnl, book, S, a, b) for a, b in shocks]
mine = p.check("taylor_pnl", mine, [p.taylor_pnl(book, S, a, b) for a, b in shocks])
pd.DataFrame({"spot move": [a for a, _ in shocks], "vol move": [b for _, b in shocks], "Taylor": mine,
              "full revaluation": [p.full_reval(book, S, a, b) for a, b in shocks]}).round(0)

In [ ]:
spot = np.array([-0.20, -0.10, -0.05, -0.02, 0.0, 0.02, 0.05, 0.10])
vol = np.array([-0.05, 0.0, 0.05, 0.10, 0.25])
full = pd.DataFrame([[p.full_reval(book, S, a, b) for b in vol] for a in spot], index=spot, columns=vol)
tay = pd.DataFrame([[p.taylor_pnl(book, S, a, b) for b in vol] for a in spot], index=spot, columns=vol)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (name, t) in zip(axes, [("full revaluation P&L", full), ("Taylor error (Taylor − full)", tay - full)]):
    im = ax.imshow(t.to_numpy() / 1000, cmap="RdBu", aspect="auto", vmin=-np.abs(t.to_numpy()).max() / 1000, vmax=np.abs(t.to_numpy()).max() / 1000)
    ax.set_xticks(range(len(vol)), [f"{v:+.0%}" for v in vol]); ax.set_yticks(range(len(spot)), [f"{s:+.0%}" for s in spot])
    ax.set(xlabel="vol shock (points / 100)", ylabel="spot shock", title=name + ", $k"); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
worst = full.stack().idxmin()
print(f"worst scenario {worst}: full revaluation {full.stack().min():,.0f}, Taylor {tay.loc[worst]:,.0f}")

The approximation is fine for small moves and badly wrong for the crash: the short puts' gamma grows as they go into the money, which a second-order expansion can't see. Risk limits use the full-revaluation grid.

## 4. The delta hedge

Flatten the book's delta with shares: the book's delta in **shares** is `Σ Δ·qty·mult` (a share has Δ = 1, multiplier 1). Trade the opposite, rounded to whole shares.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hedge_shares(book, S, r=0.045, q=0.013):
    delta_shares = 0.0
    for pos in book:
        d = 1.0 if pos.kind == "stock" else p.greeks(S, pos.K, pos.T, r, q, pos.iv, pos.cp)["delta"]
        delta_shares += ...                       # ✍️
    return -int(round(delta_shares))

mine = p.attempt(hedge_shares, book, S)
expected = -int(round(sum(p.dollar_greeks(b, S)["$delta"] for b in book) / S))
mine = p.check("hedge_shares", mine, expected)
hedged = book + [p.Position("hedge", "stock", mine, multiplier=1.0)]
print(f"trade {mine:+d} shares; book $delta after hedge: {sum(p.dollar_greeks(b, S)['$delta'] for b in hedged):,.0f}")

## 5. What a delta-hedged option earns

Buy an ATM option at an implied vol of 20% and delta-hedge it daily until expiry. What you earn depends on the vol the stock then **realizes**: roughly `Σ ½Γ·S²·(realized² − implied²)·dt`. Hedging less often adds noise; hedging costs eat the edge.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for rv in (0.15, 0.20, 0.25):
    pnl = p.delta_hedge_pnl(iv=0.20, rv=rv, steps=30)
    axes[0].hist(pnl, bins=40, alpha=0.6, label=f"realized {rv:.0%}: mean {pnl.mean():+.2f}")
axes[0].set(title="Long option at 20% IV, hedged daily", xlabel="P&L per option (spot 100)"); axes[0].legend()
rows = []
for steps in (5, 30, 120):
    for cost in (0.0, 5.0):
        pnl = p.delta_hedge_pnl(iv=0.20, rv=0.20, steps=steps, cost_bps=cost)
        rows.append({"hedges": steps, "cost (bp)": cost, "mean": pnl.mean(), "sd": pnl.std()})
t = pd.DataFrame(rows)
for cost, g in t.groupby("cost (bp)"):
    axes[1].plot(g.hedges, g["sd"], "o-", label=f"sd of P&L, cost {cost:.0f} bp")
axes[1].set(xscale="log", xlabel="hedges until expiry", title="Hedging more often: less noise, more cost"); axes[1].legend()
plt.tight_layout(); plt.show()
t.round(3)

## Wrap-up

* Greeks × quantity × multiplier; report in dollars and per vol point.
* Taylor for intuition, full revaluation for limits.
* A delta-hedged option is a bet on realized vs implied volatility; hedge frequency trades noise against cost.
* Graded versions: `labs/part06/week22_surface_portfolio` (beta-weighted book report, scenario grid, hedging simulator) and the Clinic W2 risk report.